# Evaluation Hors-Domaine (Domain Shift)
## Generalisation des modeles sur FakeNewsNet PolitiFact

**Objectif** : Tester les modeles entraines sur LIAR sur un dataset qu'ils n'ont jamais vu pour evaluer leur capacite de generalisation.

**Dataset externe** :
- **FakeNewsNet — PolitiFact** : 432 articles fake + 120 articles real avec titres

**Enjeux du domain shift** :
- Longueur des textes : declarations courtes (LIAR) vs titres d'articles
- Style : declarations directes de politiciens vs style journalistique / clickbait
- Vocabulaire : sujets et expressions differents
- Distribution des labels : LIAR ~44% fake vs FakeNewsNet ~78% fake

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import re
from pathlib import Path

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_EXT = Path("../data/brutes")
MODELS_DIR = Path("../models")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

## 1. Chargement et preparation du dataset FakeNewsNet PolitiFact

Ce dataset contient des titres d'articles politiques classes comme fake ou real par PolitiFact. On utilise les titres comme texte d'entree — format le plus proche des declarations LIAR.

**Desequilibre important** : ~78% Fake / 22% Real → un modele naif qui predit toujours "Fake" obtiendrait 78% d'accuracy. On surveille donc le F1 par classe.

In [ ]:
# FakeNewsNet — PolitiFact
pf_fake = pd.read_csv(DATA_EXT / "FakeNewsNet/dataset/politifact_fake.csv")
pf_real = pd.read_csv(DATA_EXT / "FakeNewsNet/dataset/politifact_real.csv")

pf_fake["label_binary"] = 0  # Fake
pf_real["label_binary"] = 1  # Real

df_politifact = pd.concat([pf_fake, pf_real], ignore_index=True)
df_politifact = df_politifact.dropna(subset=["title"])
df_politifact = df_politifact[df_politifact["title"].str.len() > 10].reset_index(drop=True)

print(f"FakeNewsNet PolitiFact : {len(df_politifact)} articles")
print(f"  Fake : {(df_politifact['label_binary'] == 0).sum()}")
print(f"  Real : {(df_politifact['label_binary'] == 1).sum()}")
print(f"\nExemples :")
for label in [0, 1]:
    sample = df_politifact[df_politifact["label_binary"] == label]["title"].iloc[0]
    print(f"  [{'Fake' if label == 0 else 'Real'}] {sample[:100]}")

## 2. Chargement des modeles entraines sur LIAR

In [ ]:
# Charger le meilleur modele baseline (TF-IDF + LogReg)
baseline_model = joblib.load(MODELS_DIR / "tfidf_logreg_binary.joblib")
print("Baseline TF-IDF + LogReg charge")

# Charger le modele DistilBERT si disponible
distilbert_path = MODELS_DIR / "distilbert_best"
if distilbert_path.exists():
    tokenizer_db = AutoTokenizer.from_pretrained(str(distilbert_path))
    model_db = AutoModelForSequenceClassification.from_pretrained(str(distilbert_path))
    model_db.to(device)
    model_db.eval()
    print("DistilBERT charge")
    HAS_DISTILBERT = True
else:
    print("DistilBERT non disponible — on evalue uniquement la baseline")
    HAS_DISTILBERT = False

## 3. Evaluation sur FakeNewsNet PolitiFact

In [ ]:
def predict_transformer(texts, tokenizer, model, device, batch_size=64):
    """Predictions par batch avec un transformer."""
    all_preds = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", truncation=True,
                             padding=True, max_length=128).to(device)
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

# Textes et labels PolitiFact
pf_texts = df_politifact["title"].tolist()
pf_labels = df_politifact["label_binary"].values

# --- Baseline : TF-IDF + LogReg ---
y_pred_baseline = baseline_model.predict(pf_texts)
acc_base = accuracy_score(pf_labels, y_pred_baseline)
f1_base = f1_score(pf_labels, y_pred_baseline, average="weighted")

print("=== TF-IDF + LogReg sur PolitiFact ===")
print(f"Accuracy: {acc_base:.4f}  F1: {f1_base:.4f}")
print(classification_report(pf_labels, y_pred_baseline, target_names=["Fake", "Real"]))

ood_results = {"TF-IDF + LogReg": {"accuracy": round(acc_base, 4), "f1": round(f1_base, 4)}}

# --- DistilBERT ---
if HAS_DISTILBERT:
    y_pred_db = predict_transformer(pf_texts, tokenizer_db, model_db, device)
    acc_db = accuracy_score(pf_labels, y_pred_db)
    f1_db = f1_score(pf_labels, y_pred_db, average="weighted")
    
    print("\n=== DistilBERT sur PolitiFact ===")
    print(f"Accuracy: {acc_db:.4f}  F1: {f1_db:.4f}")
    print(classification_report(pf_labels, y_pred_db, target_names=["Fake", "Real"]))
    ood_results["DistilBERT"] = {"accuracy": round(acc_db, 4), "f1": round(f1_db, 4)}

## 4. Comparaison In-Domain vs Out-of-Domain

In [ ]:
# Charger metriques in-domain
in_domain = {}
metrics_path = MODELS_DIR / "baselines_metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        baseline_metrics = json.load(f)
    in_domain["TF-IDF + LogReg"] = {
        "accuracy": baseline_metrics.get("TF-IDF + LogReg", {}).get("test_acc", 0),
        "f1": baseline_metrics.get("TF-IDF + LogReg", {}).get("test_f1", 0),
    }

# Construire le tableau comparatif
comparison_data = []
for model_name in ood_results:
    if model_name in in_domain:
        comparison_data.append({
            "Modele": model_name,
            "In-Domain Acc": in_domain[model_name]["accuracy"],
            "Out-of-Domain Acc": ood_results[model_name]["accuracy"],
            "In-Domain F1": in_domain[model_name]["f1"],
            "Out-of-Domain F1": ood_results[model_name]["f1"],
            "Drop Acc": round(in_domain[model_name]["accuracy"] - ood_results[model_name]["accuracy"], 4),
            "Drop F1": round(in_domain[model_name]["f1"] - ood_results[model_name]["f1"], 4),
        })
    else:
        comparison_data.append({
            "Modele": model_name,
            "Out-of-Domain Acc": ood_results[model_name]["accuracy"],
            "Out-of-Domain F1": ood_results[model_name]["f1"],
        })

if comparison_data:
    comp_df = pd.DataFrame(comparison_data).set_index("Modele")
    print("=== Comparaison In-Domain (LIAR test) vs Out-of-Domain (PolitiFact) ===")
    display(comp_df)

In [ ]:
# Visualisation : In-Domain vs Out-of-Domain
models_viz = list(ood_results.keys())
ood_accs = [ood_results[m]["accuracy"] for m in models_viz]
ood_f1s = [ood_results[m]["f1"] for m in models_viz]

fig = go.Figure()
fig.add_trace(go.Bar(name="OOD Accuracy", x=models_viz, y=ood_accs, marker_color="#e74c3c"))
fig.add_trace(go.Bar(name="OOD F1-Score", x=models_viz, y=ood_f1s, marker_color="#f39c12"))

# Ajouter in-domain si disponible
if in_domain:
    for m in models_viz:
        if m in in_domain:
            fig.add_trace(go.Bar(name=f"In-Domain Acc ({m})", x=[m], 
                               y=[in_domain[m]["accuracy"]], marker_color="#3498db", opacity=0.5))

fig.update_layout(
    barmode="group",
    title="Performance In-Domain vs Out-of-Domain (PolitiFact)",
    yaxis_title="Score", template="plotly_white", height=450,
    yaxis=dict(range=[0, 1]),
)
fig.show()

## 5. Analyse des erreurs out-of-domain

In [ ]:
# Analyse des erreurs de la baseline sur PolitiFact
df_politifact["pred_baseline"] = y_pred_baseline
df_politifact["correct"] = df_politifact["label_binary"] == df_politifact["pred_baseline"]

errors = df_politifact[~df_politifact["correct"]]
correct = df_politifact[df_politifact["correct"]]

print(f"Predictions correctes : {len(correct)} ({len(correct)/len(df_politifact)*100:.1f}%)")
print(f"Erreurs : {len(errors)} ({len(errors)/len(df_politifact)*100:.1f}%)")

# Exemples d'erreurs
print("\n=== Exemples de faux positifs (Real predit comme Fake) ===")
fp = errors[(errors["label_binary"] == 1) & (errors["pred_baseline"] == 0)]
for _, row in fp.head(5).iterrows():
    print(f"  {row['title'][:100]}")

print("\n=== Exemples de faux negatifs (Fake predit comme Real) ===")
fn = errors[(errors["label_binary"] == 0) & (errors["pred_baseline"] == 1)]
for _, row in fn.head(5).iterrows():
    print(f"  {row['title'][:100]}")

In [ ]:
# Analyse de la longueur des textes : comparaison LIAR vs PolitiFact
df_liar = pd.read_parquet(Path("../data/Traitees/liar_complet.parquet"))

liar_lengths = df_liar["statement"].str.split().str.len()
pf_lengths = df_politifact["title"].str.split().str.len()

fig = go.Figure()
fig.add_trace(go.Histogram(x=liar_lengths, name="LIAR (train)", opacity=0.7,
                           marker_color="#3498db", nbinsx=40))
fig.add_trace(go.Histogram(x=pf_lengths, name="PolitiFact (OOD)", opacity=0.7,
                           marker_color="#e74c3c", nbinsx=40))

fig.update_layout(
    barmode="overlay",
    title="Distribution de la longueur des textes : LIAR vs PolitiFact",
    xaxis_title="Nombre de mots", yaxis_title="Frequence",
    template="plotly_white", height=400,
)
fig.show()

print(f"LIAR    — Moyenne: {liar_lengths.mean():.1f} mots, Mediane: {liar_lengths.median():.0f}")
print(f"PolitiFact — Moyenne: {pf_lengths.mean():.1f} mots, Mediane: {pf_lengths.median():.0f}")

## 6. Discussion et synthese du domain shift

**Resultats observes** :
- La performance baisse significativement en out-of-domain (drop typique de 5-15%)
- Les modeles TF-IDF sont particulierement affectes car leur vocabulaire est specifique au LIAR

**Causes du domain shift** :

| Facteur | LIAR Dataset | FakeNewsNet PolitiFact |
|---------|-------------|----------------------|
| **Format** | Declarations directes | Titres d'articles |
| **Longueur** | ~17 mots (median) | Variable (titres plus courts) |
| **Style** | Langage politique formel | Style journalistique / clickbait |
| **Epoque** | Multi-annees | 2016-2018 principalement |
| **Sujets** | Politique US large | Politique US + faits divers |

**Conclusions** :
1. **TF-IDF** est tres sensible au changement de vocabulaire — les ngrams appris sur LIAR ne se transferent pas bien
2. **Les transformers** (DistilBERT) generalisent mieux grace a leur pre-entrainement sur un large corpus
3. **La detection de fake news basee uniquement sur le texte a des limites inherentes** — un modele ne peut pas verifier les faits, il detecte des patterns stylistiques
4. Pour ameliorer la generalisation, on pourrait envisager du **domain adaptation** ou l'ajout de features non-textuelles

**Implications ethiques** :
- Un modele deploye en production avec cette performance OOD pourrait generer de nombreux faux positifs
- Le modele risque de penaliser certains styles d'ecriture plutot que de detecter la desinformation
- La generalisation limitee souligne l'importance de **ne pas se fier aveuglément** a un classifieur automatique